# 🚦 Traffic Accident Data Analysis System
## Strategic Framework: From Descriptive Visualization to Predictive Neural Architectures

**Objective:** Analyze the US-Accidents dataset (~7.7M records, Feb 2016 – Mar 2023) to:
- Identify **high-risk locations** (Black Spots) using DBSCAN clustering
- Discover **peak accident hours** and seasonal temporal patterns
- Quantify **weather & environmental impact** on crash severity
- Build an **XGBoost severity prediction model** with SHAP explainability
- Generate **interactive geospatial heatmaps** for decision support

**Dataset:** [US-Accidents](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) by Sobhan Moosavi — covering 49 US states with 46 features including severity, GPS coordinates, timestamps, weather, and road infrastructure attributes.

---

## 1. Environment Setup and Library Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import HeatMap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score
)
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import shap

# Display settings
pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("✅ All libraries imported successfully!")

## 2. Data Acquisition and Loading

Loading the US-Accidents dataset (~7.7M records). We use a **500,000-row sample** for efficient notebook execution. Set `nrows=None` to process the full dataset.

In [ ]:
import os

# Resolve dataset path — local copy or kagglehub cache
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
LOCAL_CSV = os.path.join(DATA_DIR, "US_Accidents_March23.csv")
KAGGLE_CACHE = os.path.join(
    os.path.expanduser("~"),
    ".cache", "kagglehub", "datasets",
    "sobhanmoosavi", "us-accidents", "versions", "13",
    "US_Accidents_March23.csv",
)
DATASET_PATH = LOCAL_CSV if os.path.exists(LOCAL_CSV) else KAGGLE_CACHE

# Load dataset (sample for notebook; set nrows=None for full data)
df = pd.read_csv(DATASET_PATH, nrows=500_000)
print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
df.head()

## 3. Data Schema Inspection and Profiling

In [ ]:
# Data types overview
print("=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)

# Missing values percentage
print("\n" + "=" * 60)
print("MISSING VALUES (%)")
print("=" * 60)
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].round(2))

# Descriptive statistics
df.describe()

## 4. Data Cleaning and Missing Value Imputation

In [ ]:
print(f"Before cleaning: {df.shape}")

# Parse datetime columns
for col in ["Start_Time", "End_Time", "Weather_Timestamp"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Drop columns with >40% missing or low utility
drop_cols = [
    "End_Lat", "End_Lng", "ID", "Description",
    "Wind_Chill(F)", "Precipitation(in)", "Weather_Timestamp",
    "Turning_Loop", "Airport_Code", "Zipcode", "Country", "Number"
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors="ignore")

# Fill numeric NaN with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical NaN with mode
cat_cols = df.select_dtypes(include=["object"]).columns
for c in cat_cols:
    mode_val = df[c].mode()
    df[c] = df[c].fillna(mode_val.iloc[0] if not mode_val.empty else "Unknown")

# Validate GPS coordinates
df = df[(df["Start_Lat"].between(24, 50)) & (df["Start_Lng"].between(-130, -65))]

# Remove duplicates
df.drop_duplicates(inplace=True)

print(f"After cleaning: {df.shape}")
print(f"Remaining missing: {df.isnull().sum().sum()}")

## 5. Feature Engineering: Temporal, Spatial, and Categorical Derivations

In [ ]:
# Temporal features
df["Hour"] = df["Start_Time"].dt.hour
df["DayOfWeek"] = df["Start_Time"].dt.day_name()
df["Month"] = df["Start_Time"].dt.month
df["Year"] = df["Start_Time"].dt.year
df["Is_Weekend"] = df["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)

# Combined Place column (Street, City, State)
df["Place"] = (
    df["Street"].fillna("") + ", " +
    df["City"].fillna("") + ", " +
    df["State"].fillna("")
)

# Time of day bins
def time_of_day(h):
    if 6 <= h < 12: return "Morning"
    elif 12 <= h < 17: return "Afternoon"
    elif 17 <= h < 21: return "Evening"
    else: return "Night"

df["TimeOfDay"] = df["Hour"].apply(time_of_day)

# Duration in minutes
if "End_Time" in df.columns:
    df["Duration_min"] = (
        (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60
    ).clip(lower=0)

# Binary severity flag
df["Is_Severe"] = (df["Severity"] >= 3).astype(int)

print("New features created:")
print(df[["Hour", "DayOfWeek", "Month", "Year", "Is_Weekend", "Place", "TimeOfDay", "Is_Severe"]].head())

## 6. Exploratory Data Analysis

### 6.1 Accident Severity Distribution
The US-Accidents dataset defines severity on a 1-4 scale. Most incidents are **Severity 2** (minor), with fatal (Severity 4) crashes being extremely rare — highlighting the class imbalance challenge for ML modeling.

In [ ]:
# Severity distribution
sev = df["Severity"].value_counts().sort_index().reset_index()
sev.columns = ["Severity", "Count"]
fig = px.bar(sev, x="Severity", y="Count", color="Severity",
             title="Accident Severity Distribution",
             color_continuous_scale="OrRd",
             text="Count")
fig.update_traces(textposition="outside")
fig.update_layout(height=450)
fig.show()

# Class proportions
print("\nSeverity Class Proportions:")
print((df["Severity"].value_counts(normalize=True) * 100).round(2))

### 6.2 Temporal Distribution: Diurnal and Seasonal Patterns
Research shows accident **volume** peaks at 15:00-16:00 (commuter rush), while **severity** peaks at 12:00-13:00 and late night when higher speeds are attainable.

In [ ]:
# --- Accidents by Hour ---
hourly = df["Hour"].value_counts().sort_index().reset_index()
hourly.columns = ["Hour", "Count"]
fig = px.line(hourly, x="Hour", y="Count", markers=True,
              title="Accidents by Hour of Day",
              color_discrete_sequence=["#EF553B"])
fig.update_layout(xaxis=dict(dtick=1), height=400)
fig.show()

In [ ]:
# --- Accidents by Day of Week ---
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = df["DayOfWeek"].value_counts().reindex(day_order).reset_index()
daily.columns = ["Day", "Count"]
fig = px.bar(daily, x="Day", y="Count", title="Accidents by Day of Week",
             color_discrete_sequence=["#636EFA"])
fig.update_layout(height=400)
fig.show()

In [ ]:
# --- Monthly and Yearly Trends ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df.groupby("Month").size().plot(kind="bar", color="darkorange", ax=axes[0])
axes[0].set_title("Monthly Accident Count")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Count")

df.groupby("Year").size().plot(kind="bar", color="seagreen", ax=axes[1])
axes[1].set_title("Yearly Accident Trend")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: Hour × Day of Week ---
pivot = df.pivot_table(index="DayOfWeek", columns="Hour", values="Severity", aggfunc="count")
pivot = pivot.reindex(day_order)

fig = px.imshow(pivot, aspect="auto", color_continuous_scale="YlOrRd",
                labels=dict(color="Accidents"),
                title="Accident Heatmap: Day of Week × Hour")
fig.update_layout(height=400)
fig.show()

### 6.3 Geographic Analysis: Top States and Cities

In [ ]:
# Top 15 States
top_states = df["State"].value_counts().head(15).reset_index()
top_states.columns = ["State", "Count"]
fig = px.bar(top_states, y="State", x="Count", orientation="h",
             title="Top 15 States by Accident Count",
             color="Count", color_continuous_scale="Reds")
fig.update_layout(yaxis=dict(autorange="reversed"), height=450)
fig.show()

In [ ]:
# Top 15 Cities
top_cities = df["City"].value_counts().head(15).reset_index()
top_cities.columns = ["City", "Count"]
fig = px.bar(top_cities, y="City", x="Count", orientation="h",
             title="Top 15 Cities by Accident Count",
             color="Count", color_continuous_scale="Blues")
fig.update_layout(yaxis=dict(autorange="reversed"), height=450)
fig.show()

In [ ]:
# Day vs Night Accidents
if "Sunrise_Sunset" in df.columns:
    dn = df["Sunrise_Sunset"].value_counts().reset_index()
    dn.columns = ["Period", "Count"]
    fig = px.pie(dn, values="Count", names="Period",
                 title="Day vs Night Accidents",
                 color_discrete_sequence=["#FECB52", "#1F77B4"])
    fig.update_layout(height=400)
    fig.show()

### 6.4 Weather and Environmental Impact Analysis
Adverse weather conditions (rain, fog, snow) significantly amplify crash probability and severity. Below we analyze the top weather conditions and correlate numerical weather features with severity.

In [ ]:
# Top 15 Weather Conditions
top_weather = df["Weather_Condition"].value_counts().head(15).reset_index()
top_weather.columns = ["Condition", "Count"]
fig = px.bar(top_weather, y="Condition", x="Count", orientation="h",
             title="Top 15 Weather Conditions During Accidents",
             color="Count", color_continuous_scale="Teal")
fig.update_layout(yaxis=dict(autorange="reversed"), height=450)
fig.show()

In [ ]:
# Temperature & Visibility distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df["Temperature(F)"].dropna(), bins=60, color="salmon", edgecolor="k", alpha=0.7)
axes[0].set_title("Temperature Distribution During Accidents")
axes[0].set_xlabel("Temperature (°F)")
axes[0].set_ylabel("Frequency")

axes[1].hist(df["Visibility(mi)"].dropna(), bins=40, color="mediumpurple", edgecolor="k", alpha=0.7)
axes[1].set_title("Visibility Distribution During Accidents")
axes[1].set_xlabel("Visibility (mi)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numerical weather features
weather_features = ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
                    "Wind_Speed(mph)", "Severity"]
corr = df[weather_features].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Weather Features Correlation with Severity")
plt.tight_layout()
plt.show()

## 7. Geospatial Hotspot Mapping with Folium
Interactive heatmap showing accident density across the US. The map uses **CartoDB dark_matter** tiles for high contrast against the heat overlay.

In [ ]:
# Folium Heatmap — sampled for performance
sample = df.dropna(subset=["Start_Lat", "Start_Lng"]).sample(n=30_000, random_state=42)

m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles="CartoDB dark_matter")
heat_data = sample[["Start_Lat", "Start_Lng"]].values.tolist()
HeatMap(heat_data, radius=6, blur=10, max_zoom=13).add_to(m)
m

## 8. DBSCAN Clustering for Dynamic Black Spot Identification
Unlike K-Means, **DBSCAN** discovers clusters of arbitrary shape without requiring a predefined cluster count. Using the **haversine metric**, we identify geographic crash hotspots with a spatial radius of ~1.5 km.

In [ ]:
# DBSCAN hotspot detection
dbscan_sample = df.dropna(subset=["Start_Lat", "Start_Lng"]).sample(
    n=min(100_000, len(df)), random_state=42
)
coords = dbscan_sample[["Start_Lat", "Start_Lng"]].values

# eps in radians: km / earth_radius_km
eps_km = 1.5
eps_rad = eps_km / 6371.0
db = DBSCAN(eps=eps_rad, min_samples=50, metric="haversine", algorithm="ball_tree")
labels = db.fit_predict(np.radians(coords))

dbscan_sample = dbscan_sample.copy()
dbscan_sample["Cluster"] = labels
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f"DBSCAN Results:")
print(f"  Clusters found : {n_clusters}")
print(f"  Noise points   : {n_noise:,}")
print(f"  Clustered pts  : {(labels != -1).sum():,}")

## 9. Machine Learning: Data Preparation

We select **temporal**, **weather**, **road infrastructure** (boolean), and **environmental** features for severity classification. The target is `Severity` (1-4), zero-indexed for XGBoost compatibility.

In [ ]:
# Feature selection
feature_cols = [
    "Hour", "Month", "Is_Weekend",
    "Temperature(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Distance(mi)",
]
bool_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal",
]
cat_encode_cols = ["Sunrise_Sunset"]

# Build feature matrix
all_cols = feature_cols + bool_cols + cat_encode_cols + ["Severity"]
ml_df = df[[c for c in all_cols if c in df.columns]].dropna().copy()

# Label-encode categoricals
le_map = {}
for c in cat_encode_cols:
    if c in ml_df.columns:
        le = LabelEncoder()
        ml_df[c] = le.fit_transform(ml_df[c].astype(str))
        le_map[c] = le

X_cols = [c for c in feature_cols + bool_cols + cat_encode_cols if c in ml_df.columns]
X = ml_df[X_cols].astype(float)
y = ml_df["Severity"] - 1  # 0-indexed

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Feature matrix: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nClass distribution (train):")
print(y_train.value_counts().sort_index())

## 10. Baseline Models: Logistic Regression & Random Forest

In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=500, multi_class="multinomial", random_state=42, n_jobs=-1)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred_lr, average='weighted'):.4f}")

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\n=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

## 11. XGBoost for Accident Severity Classification

XGBoost minimizes:

$$\mathcal{L}(\phi) = \sum_i l(\hat{y}_i, y_i) + \sum_k \Omega(f_k)$$

where $l$ is a differentiable convex loss and $\Omega$ regularizes tree complexity using both first-order (gradient) and second-order (Hessian) derivatives for efficient split finding.

In [ ]:
# XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=int(y.nunique()),
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred_xgb = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb, average="weighted")

print(f"\n=== XGBoost Results ===")
print(f"Accuracy : {acc_xgb:.4f}")
print(f"F1 (weighted): {f1_xgb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

## 12. Model Evaluation: Confusion Matrix & Feature Importance

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("XGBoost Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# Feature Importance (Gain)
xgb.plot_importance(xgb_model, ax=axes[1], max_num_features=15,
                    importance_type="gain", color="steelblue")
axes[1].set_title("XGBoost Feature Importance (Gain)")

plt.tight_layout()
plt.show()

In [ ]:
# Model comparison table
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        acc_xgb,
    ],
    "F1 (weighted)": [
        f1_score(y_test, y_pred_lr, average="weighted"),
        f1_score(y_test, y_pred_rf, average="weighted"),
        f1_xgb,
    ],
}).round(4)
print("=== Model Comparison ===")
results

## 13. Explainable AI with SHAP

SHAP (SHapley Additive exPlanations) assigns a **contribution score** to every feature for each prediction, grounded in cooperative game theory. This transparency is critical for translating ML outputs into actionable, legally defensible policy interventions.

In [ ]:
# SHAP Explainability
explainer = shap.TreeExplainer(xgb_model)
shap_sample = X_test.iloc[:2000]
shap_values = explainer.shap_values(shap_sample)

# Beeswarm summary plot
print("SHAP Summary Plot (Beeswarm):")
shap.summary_plot(shap_values, shap_sample, max_display=15, show=True)

In [ ]:
# SHAP Bar plot — mean absolute SHAP values
print("SHAP Feature Importance (Bar):")
shap.summary_plot(shap_values, shap_sample, plot_type="bar", max_display=15, show=True)

## 14. Key Insights and Conclusions

### Temporal Patterns
- **Peak accident hours** occur during afternoon/evening rush (14:00–17:00), while **severity peaks** at midday and late night when higher speeds prevail
- **Weekday** accidents significantly outnumber weekends due to commuter traffic

### Geographic Hotspots
- **California, Florida, and Texas** consistently lead in accident counts
- Large metro areas (Miami, Houston, Los Angeles) dominate city-level statistics
- DBSCAN identified dense crash clusters corresponding to major highway interchanges

### Weather Impact
- **Fair/Clear** weather accounts for the majority of accidents (sheer exposure volume), but **Rain, Fog, and Snow** disproportionately elevate severity
- Lower **visibility** and higher **humidity** show moderate positive correlation with severity

### Model Performance
- **XGBoost outperforms** Logistic Regression and Random Forest on this tabular dataset
- Class imbalance remains a challenge — Severity 1 and 4 are significantly underrepresented

### SHAP Explainability
- **Hour of day**, **Distance**, **Visibility**, and **Temperature** are the most influential features
- Road infrastructure features (Traffic_Signal, Crossing) have meaningful but secondary contributions
- These insights enable **evidence-based** infrastructure investment and dynamic speed limit policies

---
**Next Steps:**
- Apply SMOTE/Focal Loss to improve minority-class recall
- Integrate GNN architectures with OSMnx road graphs for topological modeling
- Deploy real-time CV pipeline (YOLO + DeepSORT) for TTDA minimization
- Build Streamlit dashboard → `streamlit run dashboard/app.py`